In [13]:
import pandas as pd

multi = pd.read_csv(r"D:\AQI_project\CAMS\cams_multi_level_integrated.csv",parse_dates = ["time", "valid_time"])

single = pd.read_csv(r"D:\AQI_project\CAMS\cams_single_level_integrated.csv",parse_dates = ["time", "valid_time"])

In [14]:
print("multi-level :", multi.shape)
print("single-level :", single.shape)

multi-level : (29224, 20)
single-level : (29224, 12)


In [15]:
print("Multi-level columns : ",)
print( multi.columns.tolist())

print("Single-level columns : ",)
print( single.columns.tolist())

Multi-level columns : 
['time', 'step', 'hybrid', 'latitude', 'longitude', 'valid_time', 'co', 'aermr04', 'aermr05', 'aermr06', 'aermr09', 'aermr07', 'aermr10', 'aermr08', 'no2', 'no', 'go3', 'q', 'so2', 't']
Single-level columns : 
['time', 'number', 'step', 'surface', 'latitude', 'longitude', 'valid_time', 'u10', 'v10', 'd2m', 't2m', 'sp']


In [16]:
# the common identifying columns are time, step, lat,lon, valid_time
# make sure each dataset has one row for each combination of those columns


keys = ["time","step","latitude","longitude","valid_time"]

print("Multi-duplicates : ", multi.duplicated(keys).sum())
print("Single-duplicates : ", single.duplicated(keys).sum())

Multi-duplicates :  0
Single-duplicates :  0


In [17]:
# combining single level and multi level cams data

cams_merged = pd.merge(
    multi,
    single,
    on = keys,
    how = "inner"
)

print(cams_merged.shape)
print(cams_merged.head())

(29224, 27)
                 time    step  hybrid  latitude  longitude  \
0 2016-01-01 00:00:00  0 days    60.0      28.5      77.25   
1 2016-01-01 03:00:00  0 days    60.0      28.5      77.25   
2 2016-01-01 06:00:00  0 days    60.0      28.5      77.25   
3 2016-01-01 09:00:00  0 days    60.0      28.5      77.25   
4 2016-01-01 12:00:00  0 days    60.0      28.5      77.25   

           valid_time        co       aermr04       aermr05       aermr06  \
0 2016-01-01 00:00:00  0.000005  7.207745e-11  8.731149e-11  9.958967e-11   
1 2016-01-01 03:00:00  0.000004  8.026291e-11  1.004992e-10  1.323315e-10   
2 2016-01-01 06:00:00  0.000002  1.423359e-10  2.480647e-10  5.851462e-10   
3 2016-01-01 09:00:00  0.000001  4.804406e-10  9.089263e-10  9.210908e-10   
4 2016-01-01 12:00:00  0.000002  4.935146e-10  8.035954e-10  1.299441e-10   

   ...         q           so2          t  number  surface       u10  \
0  ...  0.006776  1.499882e-07  282.65347       0      0.0  1.443085   
1  ...  

In [18]:
# conversion from UTC to IST
# cams_merged has to integrate with CPCB data

cams_merged["time"] = cams_merged["time"] + pd.Timedelta(hours=5, minutes=30)
cams_merged["valid_time"] = cams_merged["valid_time"] + pd.Timedelta(hours=5, minutes=30)

print(cams_merged[["time", "valid_time"]].head())

                 time          valid_time
0 2016-01-01 05:30:00 2016-01-01 05:30:00
1 2016-01-01 08:30:00 2016-01-01 08:30:00
2 2016-01-01 11:30:00 2016-01-01 11:30:00
3 2016-01-01 14:30:00 2016-01-01 14:30:00
4 2016-01-01 17:30:00 2016-01-01 17:30:00


In [19]:
# CPCB data concantination 

import pandas as pd
from pathlib import Path

cpcb_folder = Path(r"C:\Users\admin\Documents\My_projects\AQI_project\data\raw\CPCB")

cpcb_dfs = []

for file_path in cpcb_folder.glob("*.csv"):
    print("Processing:", file_path.name)
    
    df = pd.read_csv(file_path)
    cpcb_dfs.append(df)

cpcb_combined = pd.concat(
    cpcb_dfs,
    ignore_index=True
)

print("Shape:", cpcb_combined.shape)
print(cpcb_combined.head())



Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H(1).csv
Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H(2).csv
Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H(3).csv
Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H(4).csv
Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H(5).csv
Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H(6).csv
Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H(7).csv
Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H(8).csv
Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H(9).csv
Processing: raw_data_hourly_anand_vihar,_delhi_-_dpcc_1H.csv
Shape: (87672, 25)
             Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01 00:00:00         160.75        262.25       25.12        44.80   
1  2024-01-01 01:00:00         170.50        254.00       22.10        49.28   
2  2024-01-01 02:00:00         172.50        243.25       13.68        45.20   
3  2024-01-01 03:00:00  

In [20]:
cpcb_combined["Timestamp"] = pd.to_datetime(cpcb_combined["Timestamp"])

cpcb_combined = cpcb_combined.sort_values("Timestamp").reset_index(drop=True)

In [21]:
print(cpcb_combined.columns.tolist())
print(cpcb_combined.info())

['Timestamp', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)', 'O Xylene (µg/m³)', 'Eth-Benzene (µg/m³)', 'MP-Xylene (µg/m³)', 'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)', 'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)']
<class 'pandas.DataFrame'>
RangeIndex: 87672 entries, 0 to 87671
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Timestamp            87672 non-null  datetime64[us]
 1   PM2.5 (µg/m³)        74825 non-null  float64       
 2   PM10 (µg/m³)         72359 non-null  float64       
 3   NO (µg/m³)           69660 non-null  float64       
 4   NO2 (µg/m³)          72863 non-null  float64       
 5   NOx (ppb)            70097 non-null  float64       
 6   NH3 (µg/m³)          71754 non-null  float64       
 7   SO2

In [22]:
print(cpcb_combined["Timestamp"].head(10))
print(cpcb_combined["Timestamp"].tail(10))

0   2016-01-01 00:00:00
1   2016-01-01 01:00:00
2   2016-01-01 02:00:00
3   2016-01-01 03:00:00
4   2016-01-01 04:00:00
5   2016-01-01 05:00:00
6   2016-01-01 06:00:00
7   2016-01-01 07:00:00
8   2016-01-01 08:00:00
9   2016-01-01 09:00:00
Name: Timestamp, dtype: datetime64[us]
87662   2025-12-31 14:00:00
87663   2025-12-31 15:00:00
87664   2025-12-31 16:00:00
87665   2025-12-31 17:00:00
87666   2025-12-31 18:00:00
87667   2025-12-31 19:00:00
87668   2025-12-31 20:00:00
87669   2025-12-31 21:00:00
87670   2025-12-31 22:00:00
87671   2025-12-31 23:00:00
Name: Timestamp, dtype: datetime64[us]


In [24]:
cpcb_combined["Timestamp"] = pd.to_datetime(cpcb_combined["Timestamp"])

print(cpcb_combined["Timestamp"].min())
print(cpcb_combined["Timestamp"].max())

2016-01-01 00:00:00
2025-12-31 23:00:00


In [25]:
# to check whether any hourly timestamps are missing or duplicated

print("Duplicate timestamps:", cpcb_combined["Timestamp"].duplicated().sum())

expected = pd.date_range(
    start=cpcb_combined["Timestamp"].min(),
    end=cpcb_combined["Timestamp"].max(),
    freq="h"
)

print("Expected rows:", len(expected))
print("Actual rows:", len(cpcb_combined))

Duplicate timestamps: 0
Expected rows: 87672
Actual rows: 87672


In [26]:
# to check CAMS start / end and timestamp type before merge

print(cams_merged["time"].min())
print(cams_merged["time"].max())
print(cams_merged["time"].dtype)

2016-01-01 05:30:00
2026-01-01 02:30:00
datetime64[us]


In [27]:
print(cams_merged[["time"]].head(10))

                 time
0 2016-01-01 05:30:00
1 2016-01-01 08:30:00
2 2016-01-01 11:30:00
3 2016-01-01 14:30:00
4 2016-01-01 17:30:00
5 2016-01-01 20:30:00
6 2016-01-01 23:30:00
7 2016-01-02 02:30:00
8 2016-01-02 05:30:00
9 2016-01-02 08:30:00


In [29]:
# interpolation of CPCB data

cpcb_interp = cpcb_combined.copy()

cpcb_interp = cpcb_interp.set_index("Timestamp")

cpcb_interp = cpcb_interp.resample("30min").asfreq()

print(cpcb_interp[["PM2.5 (µg/m³)"]].head(10))

                     PM2.5 (µg/m³)
Timestamp                         
2016-01-01 00:00:00         389.33
2016-01-01 00:30:00            NaN
2016-01-01 01:00:00         376.00
2016-01-01 01:30:00            NaN
2016-01-01 02:00:00         480.50
2016-01-01 02:30:00            NaN
2016-01-01 03:00:00         486.67
2016-01-01 03:30:00            NaN
2016-01-01 04:00:00         441.17
2016-01-01 04:30:00            NaN


In [30]:
cpcb_interp = cpcb_interp.interpolate(method="time")
print(cpcb_interp[["PM2.5 (µg/m³)"]].head(10))

                     PM2.5 (µg/m³)
Timestamp                         
2016-01-01 00:00:00        389.330
2016-01-01 00:30:00        382.665
2016-01-01 01:00:00        376.000
2016-01-01 01:30:00        428.250
2016-01-01 02:00:00        480.500
2016-01-01 02:30:00        483.585
2016-01-01 03:00:00        486.670
2016-01-01 03:30:00        463.920
2016-01-01 04:00:00        441.170
2016-01-01 04:30:00        518.000


In [31]:
cpcb_interp = cpcb_interp.reset_index()
print(cpcb_interp[["Timestamp", "PM2.5 (µg/m³)"]].head())

            Timestamp  PM2.5 (µg/m³)
0 2016-01-01 00:00:00        389.330
1 2016-01-01 00:30:00        382.665
2 2016-01-01 01:00:00        376.000
3 2016-01-01 01:30:00        428.250
4 2016-01-01 02:00:00        480.500


In [32]:
# to check whether the two datasets have matching timestamps

print("CPCB start:", cpcb_interp["Timestamp"].min())
print("CPCB end:", cpcb_interp["Timestamp"].max())

print("CAMS start:", cams_merged["time"].min())
print("CAMS end:", cams_merged["time"].max())

CPCB start: 2016-01-01 00:00:00
CPCB end: 2025-12-31 23:00:00
CAMS start: 2016-01-01 05:30:00
CAMS end: 2026-01-01 02:30:00


In [33]:
# to check actual matching timestamps

cpcb_times = set(cpcb_interp["Timestamp"])
cams_times = set(cams_merged["time"])

common_times = cpcb_times.intersection(cams_times)

print("CPCB timestamps:", len(cpcb_times))
print("CAMS timestamps:", len(cams_times))
print("Matching timestamps:", len(common_times))

CPCB timestamps: 175343
CAMS timestamps: 29224
Matching timestamps: 29222


In [34]:
# to identify 2 unmatched CAMS timestamps

unmatched_cams = sorted(cams_times - cpcb_times)

print("Unmatched CAMS timestamps:")
print(unmatched_cams)

Unmatched CAMS timestamps:
[Timestamp('2025-12-31 23:30:00'), Timestamp('2026-01-01 02:30:00')]


In [35]:
### FINAL DATA

final_data = pd.merge(
    cpcb_interp,
    cams_merged,
    left_on="Timestamp",
    right_on="time",
    how="inner"
)

print(final_data.shape)
print(final_data.head())
print(final_data.tail())

(29222, 52)
            Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0 2016-01-01 05:30:00        518.080        801.00     246.330       93.315   
1 2016-01-01 08:30:00        387.000        619.17      86.055      104.780   
2 2016-01-01 11:30:00        234.835        505.50      80.020      131.675   
3 2016-01-01 14:30:00        144.420        440.17      80.415      120.425   
4 2016-01-01 17:30:00        254.585        839.25     351.155      181.920   

    NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  ...  \
0  358.415000       159.25       15.730       1.770          7.850  ...   
1  210.895000       102.32       30.520       2.035          8.360  ...   
2  226.610000       102.51       96.855       1.370         24.685  ...   
3  217.060000        66.36       27.870       0.540         27.355  ...   
4  257.191875        92.25       36.875       3.530         11.295  ...   

          q           so2          t  number  surface       u1

In [37]:
final_data.to_csv(r"C:\Users\admin\Documents\My_projects\AQI_project\data\final_integrated_data.csv", index=False)
print("Saved successfully!")
print(final_data.shape)

Saved successfully!
(29222, 52)
